# 00 — Data ingestion & cleaning

Pull Indian equity OHLCV from the HuggingFace dataset
`vishnun0027/indian-market-historical-ohlcv`, clean it, and look hard at what
arrived before building anything on it.

Standalone: this notebook does not import the `portfolio_agent` package. It needs
only `afa_lab.py`, which sits next to it.

**Run this first.** It populates `data_cache/`, which every other notebook reuses,
so the download happens once.

## Setup

In [ ]:
# Dependencies. Torch is only needed by the two learned strategies (04, 05).
# !pip install -q pandas numpy pyarrow matplotlib huggingface_hub torch

import sys, pathlib

# afa_lab.py sits next to this notebook. On Colab (or anywhere the file is
# missing) fetch it from the repo — that is the only network call that touches
# GitHub, and nothing else here imports the portfolio_agent package.
if not pathlib.Path("afa_lab.py").exists():
    import urllib.request
    URL = ("https://raw.githubusercontent.com/3dwag98/afa/main/"
           "notebooks/standalone/afa_lab.py")
    urllib.request.urlretrieve(URL, "afa_lab.py")
    print("fetched afa_lab.py")

sys.path.insert(0, ".")
import afa_lab as L

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
print("toolkit loaded | torch available:", L.TORCH_AVAILABLE)

## Universe

In [ ]:
# NSE large caps. Any symbol absent from the dataset is skipped rather than
# failing the run, so this list does not have to be exactly right.
UNIVERSE = [
    "RELIANCE",
    "TCS",
    "HDFCBANK",
    "INFY",
    "ICICIBANK",
    "HINDUNILVR",
    "ITC",
    "SBIN",
    "BHARTIARTL",
    "KOTAKBANK",
    "LT",
    "AXISBANK",
    "ASIANPAINT",
    "MARUTI",
    "SUNPHARMA",
    "TITAN",
    "ULTRACEMCO",
    "WIPRO",
    "NESTLEIND",
    "BAJFINANCE",
    "TATAMOTORS",
    "TATASTEEL",
    "POWERGRID",
    "NTPC",
    "ONGC",
    "HCLTECH",
    "JSWSTEEL",
    "GRASIM",
    "CIPLA",
    "COALINDIA"
]

START_DATE = "2018-01-01"
END_DATE   = None          # None = up to the dataset's last session
CACHE      = "data_cache"  # downloaded parquet files land here and are reused

print(len(UNIVERSE), "symbols requested")

## Ingest

The dataset carries `date, open, high, low, close, adj_close, volume,
dividends, stock_splits, symbol` per file, one file per symbol under `stocks/`.
Indices such as `^NSEI` live under `indices/`.

In [ ]:
# Ingestion. One small parquet per symbol is pulled from the Hub dataset
# `vishnun0027/indian-market-historical-ohlcv` (2,421 NSE/BSE equities) and
# cleaned. Downloading per symbol rather than snapshotting the repo means a
# 30-name universe fetches 30 small files instead of 283 MB.
#
# Cleaning, in order: back-adjust OHLC by adj_close/close so a split is not read
# as a 90% crash, coerce numerics, drop unparseable dates and missing closes
# (rather than forward-filling, so a gap stays visible), and drop duplicate
# sessions keeping the last.
#
# If the Hub is unreachable the toolkit falls back to a synthetic panel and says
# so loudly. Synthetic results describe the generator, not the market.

panel = L.load_panel(UNIVERSE, start_date=START_DATE, end_date=END_DATE,
                     cache_dir=CACHE)

close = L.align_close_matrix(panel)
print(f"{len(panel)} symbols | {close.index.min().date()} -> {close.index.max().date()}"
      f" | {len(close)} sessions")

## What cleaning actually did

The single most consequential step is the price adjustment. On a 1:10 split the
raw close drops 90% in one print — cross-sectional momentum reads that as a
crash, and every ATR-derived stop built on it blows out. Back-adjusting all four
price legs by the same `adj_close / close` factor removes the discontinuity
while leaving intraday relationships intact: a locked session (high == low)
stays locked.

In [ ]:
symbol = sorted(panel)[0]
frame = panel[symbol]

print(f"{symbol}: {len(frame)} sessions, {frame.index.min().date()} -> {frame.index.max().date()}")
display(frame.head(3))
display(frame.describe().T[["mean", "std", "min", "max"]])

returns = frame["close"].pct_change()
print("\nlargest single-day moves (a split that slipped through the adjustment "
      "would show up here as a ~-90% day):")
display(returns.abs().nlargest(5))

## Quality

Coverage below ~95% of business days means the symbol was suspended, newly
listed, or partially missing from the dataset — and a cross-sectional strategy
that ranks it against fully-covered names is comparing different things.

In [ ]:
quality = L.panel_quality(panel)
display(quality.sort_values("coverage").head(10))

L.plot_data_quality(quality)
L.plot_prices(panel, n=8)

## Features

In [ ]:
# Features are computed per symbol and left NaN until each window has filled.
# They are never back-filled: a back-filled indicator is a look-ahead, and it is
# invisible in every metric downstream.
feature_panel = L.build_feature_panel(panel)

sample = feature_panel[sorted(feature_panel)[0]]
print(f"{len(sample.columns)} features:", list(sample.columns))
display(sample.dropna().tail(3))

In [ ]:
# How much history each feature needs before it produces anything. Long-lookback
# features are why a full trading year of warm-up is the default.
warmup = {c: int(sample[c].isna().sum()) for c in sample.columns}
display(pd.Series(warmup, name="leading NaNs").sort_values(ascending=False).head(10))

In [ ]:
# Cross-sectional dispersion: how much the ranked features actually separate
# names on a given date. A feature with no dispersion cannot drive a ranking.
import matplotlib.pyplot as plt

interesting = ["mom_9m_skip1m", "realized_vol_60", "rsi_14", "breakout_20"]
fig, axes = plt.subplots(1, len(interesting), figsize=(4 * len(interesting), 3.2))
for ax, name in zip(axes, interesting):
    matrix = L.cross_section(feature_panel, name)
    spread = matrix.std(axis=1)
    ax.plot(spread.index, spread, color=L.PALETTE[0], linewidth=1.1)
    ax.set_title(f"{name}\ncross-sectional dispersion", fontsize=9)
    ax.grid(**L.GRID)
plt.tight_layout(); plt.show()

## Next

`data_cache/` now holds the cleaned panel. The strategy notebooks reuse it:

| Notebook | Strategy |
| --- | --- |
| `01_rule_based.ipynb` | trend + breakout + volume + Monte Carlo composite |
| `02_momentum.ipynb` | cross-sectional momentum with a crash filter |
| `03_low_volatility.ipynb` | inverse-volatility weighted low-vol |
| `04_lstm.ipynb` | supervised LSTM on cross-sectional forward-return rank |
| `05_sac_rl.ipynb` | SAC reinforcement-learning allocation policy |
| `06_ensemble_comparison.ipynb` | all of them, blended and compared |

---

## What this does and does not show

Read before quoting any number above.

- **Survivorship.** The universe is today's large caps, applied to history. Names
  that were large caps in 2018 and are not now are absent, and they are absent
  precisely because they did badly. Every long-only result here is biased upward
  by an amount this notebook cannot measure. A point-in-time constituent list is
  the only fix, and this dataset does not carry one.
- **One universe, one period.** Thirty names over a few years is a single draw.
  The difference between two strategies here is well within what the draw alone
  could produce.
- **Costs are a flat 25 bps.** Real cost scales with size and with how illiquid
  the name is, and the fill is assumed at the close. A strategy whose edge is
  this side of costs is not distinguishable from one that has no edge.
- **No point-in-time fundamentals, no corporate actions beyond the price
  adjustment**, and no circuit-limit modelling. On Indian equities a
  circuit-locked session is untradeable, and the simulation will happily trade it.
- **Parameters were chosen, not fitted.** Nothing here is tuned on a held-out
  period. That is deliberate — tuning on this sample and reporting the result
  would be reporting the tuning.

The purpose of these notebooks is to make the mechanism legible and modifiable,
not to establish that any of these strategies makes money.